# Tutorial: Delegated Auth for Trustworthy Power BI Delivery

Audience:
- Power BI and Fabric presenters
- BI developers and engineers who want the safest live-demo path

Prerequisites:
- notebook kernel points at the repo `.venv`
- `.env` filled in locally with `AUTH_MODE=delegated` or `PBI_AUTH_MODE=delegated`
- manual prerequisites completed in `docs/operations/local-setup.md`
- signed-in user has workspace access and dataset Build for `executeQueries`

Learning goals:
- load local configuration safely
- acquire a delegated Power BI token with device code flow
- list workspaces, datasets, and reports in real user context
- run a small DAX query as a validation-friendly probe


## Outline

1. Load config
2. Acquire a delegated token
3. List workspaces
4. List datasets and reports
5. Execute a small DAX query

This notebook uses device code flow by default because it is the least fragile local demo path and the closest fit to trustworthy user-context validation.

In [ ]:
from __future__ import annotations

import pandas as pd

from src.config.loader import require_dataset_id, require_group_id
from src.notebooksupport import bootstrap_notebook

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)


## Step 1 - Load config

This reads `.env`, validates the required values, and shows a redacted summary so secrets are never printed. If your default config is not delegated, the notebook will still force delegated auth for its own calls.

In [ ]:
context = bootstrap_notebook()
config = context.settings
effective_auth_mode = "delegated"

summary = config.redacted_summary()
summary["effective_auth_mode"] = effective_auth_mode
summary


## Step 2 - Acquire a delegated token

This uses the configured delegated auth flow. If device code is blocked because the app registration is confidential, the auth helper can fall back to browser sign-in when `CLIENT_SECRET` and `REDIRECT_URI` are configured.

In [ ]:
client = context.create_client(auth_mode=effective_auth_mode, use_device_code=config.use_device_code)
print("Delegated access token acquired.")


## Step 3 - List workspaces

This shows the workspaces visible to the signed-in user. If your target workspace is missing, the problem is almost always user access or tenant policy, not the notebook code.

In [ ]:
workspaces_df = pd.DataFrame(client.get_groups())
workspaces_df


## Step 4 - Select the workspace

The notebook uses `WORKSPACE_ID` from `.env`. You can override it manually in the next cell if you want to pick a different workspace from the table above.

In [ ]:
group_id = require_group_id(None, config)
group_id


## Step 5 - List datasets and reports

These calls depend on workspace access. If datasets list correctly but query execution fails later, check dataset Build permission first.

In [ ]:
datasets_df = pd.DataFrame(client.get_datasets_in_group(group_id))
reports_df = pd.DataFrame(client.get_reports_in_group(group_id))

print("Datasets")
display(datasets_df)
print("Reports")
display(reports_df)


## Step 6 - Execute a very small DAX query

This step depends on dataset Read and Build permissions. Keep the query short and presentation-friendly.

In [ ]:
dataset_id = require_dataset_id(None, config)
query_df = pd.DataFrame(
    client.execute_queries_in_group(
        group_id=group_id,
        dataset_id=dataset_id,
        dax_query=config.dax_query,
        impersonated_user_name=config.impersonated_user_name,
    )
)
query_df


## Presenter notes

- This is the recommended demo path.
- Device code flow avoids redirect URI setup friction.
- The signed-in user sees only what they can really access.
- If `executeQueries` fails, mention dataset Build permission and tenant settings before changing code.
